# Python code development for BC Indigenous Business Analysis

In [1]:
import diversedata as dd
import pandas as pd
import altair as alt
from IPython.display import Markdown

## 1. Data Cleaning & Processing

In [2]:
# use kable for R

In [3]:
# Load and clean the data
indigenousbiz_data = dd.load_data("bcindigenousbiz")
indigenousbiz_data = indigenousbiz_data.dropna(subset=["latitude", "longitude"])
indigenousbiz_data["region"] = indigenousbiz_data["region"].str.title()
indigenousbiz_data["industry_sector"] = pd.Categorical(
    indigenousbiz_data["industry_sector"], ordered=False
)
indigenousbiz_data["year_formed"] = indigenousbiz_data["year_formed"].astype("Int64")

# Preview cleaned data
Markdown(indigenousbiz_data.head().to_markdown())

|    | business_name                                          | city         |   latitude |   longitude | region                     | type                    | industry_sector                               |   year_formed | number_of_employees   |
|---:|:-------------------------------------------------------|:-------------|-----------:|------------:|:---------------------------|:------------------------|:----------------------------------------------|--------------:|:----------------------|
|  0 | Ellipsis Energy Inc                                    | Moberly Lake |    55.8194 |    -121.835 | Northeast                  | Private Company         | Mining, quarrying, and oil and gas extraction |          2012 | 5 to 9                |
|  1 | Indigenous Community Development & Prosperity (ICDPRO) | Enderby      |    50.5515 |    -119.134 | Thompson / Okanagan        | Private Company         | Other services (except public administration) |          2020 | 1 to 4                |
|  2 | Formline Construction Ltd.                             | Burnaby      |    49.2661 |     123.006 | Lower Mainland / Southwest | Private Company         | Construction                                  |          2021 | 1 to 4                |
|  3 | Quilakwa Investments Ltd.                              | Enderby      |    50.5375 |    -119.142 | Thompson / Okanagan        | Community Owned Company | Accommodation and food services               |          1984 | 20 to 49              |
|  4 | Quilakwa Esso                                          | Enderby      |    50.5375 |    -119.142 | Thompson / Okanagan        | Community Owned Company | Retail trade                                  |          1984 | 10 to 19              |

## 2. Exploratory Data Analysis (EDA)

Business count by Region

In [4]:
# remove R legend/fill and bar order

In [5]:
alt.Chart(indigenousbiz_data).mark_bar().encode(
    x=alt.X("count()").title("Number of Businesses"),
    y=alt.Y("region").sort("x").title("Region"),
).properties(title="Business Count per Region", width=500, height=300)

alt.Chart(...)

Mapping Businesses

Industry Analysis

In [6]:
# remove R legend/fill and bar order and show code

In [7]:
top_10_sectors = (
    indigenousbiz_data.groupby("industry_sector", observed=True)
    .size()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
    .head(10)
)

alt.Chart(top_10_sectors).mark_bar().encode(
    x=alt.X("count").title("Number of Businesses"),
    y=alt.Y("industry_sector").sort("x").title("Sector").axis(alt.Axis(labelLimit=300)),
).properties(title="Top Industry Sectors", width=500, height=300)

alt.Chart(...)

## 3. Chi-Squared Test of Independence

In [ ]:
# silence R warning

In [10]:
from scipy.stats import chi2_contingency

# Step 1: Create contingency table of 'type' vs 'region'
type_region_table = pd.crosstab(indigenousbiz_data['type'], indigenousbiz_data['region'])

# Perform chi-square test of independence
chi2_stat, p_value, dof, expected = chi2_contingency(type_region_table)

# Find minimum expected count
min_expected = expected.min()

print("Chi-Square Assumptions:")
print(f"- Minimum expected count: {min_expected:.2f}")

if min_expected >= 5:
    print("Chi-squared assumptions met. Proceeding with standard test.\n")
    print(f"Chi2 Statistic: {chi2_stat}")
    print(f"P-value: {p_value}")
    print(f"Degrees of Freedom: {dof}")
else:
    print("Assumptions violated. Proceeding with simulation-based Chi-squared test.\n")

Chi-Square Assumptions:
- Minimum expected count: 0.01
Assumptions violated. Proceeding with simulation-based Chi-squared test.



In [12]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import altair as alt

# Drop missing values in the columns of interest
df = indigenousbiz_data.dropna(subset=['type', 'region'])

# Function to calculate Chi-square statistic
def chi2_statistic(data):
    table = pd.crosstab(data['type'], data['region'])
    chi2, _, _, _ = chi2_contingency(table)
    return chi2

# 1. Observed statistic
observed_stat = chi2_statistic(df)

# 2. Generate null distribution via permutation
n_reps = 1000
null_distribution = []

for _ in range(n_reps):
    shuffled = df.copy()
    shuffled['region'] = np.random.permutation(shuffled['region'])
    null_distribution.append(chi2_statistic(shuffled))

null_distribution = np.array(null_distribution)

# 3. Calculate simulation-based p-value
p_value = np.mean(null_distribution >= observed_stat)

print(f"Simulation-based p-value: {p_value:.4e}")

# 4. Plot permutation distribution with shading
null_df = pd.DataFrame({'chi2_stat': null_distribution})

# Base histogram
hist = alt.Chart(null_df).mark_bar(color='lightgray').encode(
    alt.X('chi2_stat:Q', bin=alt.Bin(maxbins=50), title='Chi-square statistic'),
    alt.Y('count()', title='Frequency')
)

# Shaded area for p-value
shade_df = null_df[null_df['chi2_stat'] >= observed_stat]
shade = alt.Chart(shade_df).mark_bar(color='red').encode(
    alt.X('chi2_stat:Q', bin=alt.Bin(maxbins=50)),
    alt.Y('count()')
)

# Vertical line for observed statistic
line = alt.Chart(pd.DataFrame({'obs': [observed_stat]})).mark_rule(color='black').encode(
    x='obs:Q'
)

# Combine charts
chart = hist + shade + line
chart.properties(
    width=600,
    height=400,
    title='Permutation-based Null Distribution (Chi-square)'
)


Simulation-based p-value: 0.0000e+00


alt.LayerChart(...)